In [ ]:
#%matplotlib inline
%time from hikyuu.interactive import *

# 1 Use TM to implement a simple account book

The TradeManager object can be understood as a simulated trading account, responsible for the buy/sell operations, recording the trade records and the positions; it can also be connected to the live trading by modifying the buy/sell operation interfaces. To create a simulated trading account, the quick creation function crtTM is usually used. The basic operations of a TM object:

- buy  buy
- sell  sell
- checkin  deposit cash
- checkout  withdraw cash

TM can be used to implement a simple account book to manually record your own operations, e.g.:

In [ ]:
# Create a simulated account with an initial capital of 100,000, starting from 2017-01-01
my_tm = crtTM(init_cash=100000, date=Datetime(201701010000))

# On 2017-01-03, buy 100 shares at the price of 9.11
td = my_tm.buy(Datetime(201701030000), sm['sz000001'], 9.11, 100)

# View the current cash and position
print(my_tm)

In [ ]:
# Convert to a pandas DataFrame to display the current positions 
position = my_tm.get_position_list()
position.to_df()

In [ ]:
my_tm.get_trade_list().to_df()

In [ ]:
# On 2017-02-21, sell 100 shares at the price of 9.60
td = my_tm.sell(Datetime(201702210000), sm['sz000001'], 9.60)

my_tm

# 2 View the trade details with Excel

Use the tocsv method to save the trade records, the current positions and the closed-trade details of the TM to csv files respectively, so as to view the details with Excel.

The parameter of the tocsv method is a specified directory, which must exist. Its output generates three files in the specified directory: "TM name_trade records.csv", "TM name_open positions.csv", "TM name_closed trades.csv". The TM name can be specified when creating the TM object with crtTM, and defaults to "SYS", as shown in the following figure.

<img src="images/008_01_tocsv.png" align='left'>

In [ ]:
# Output to the temporary path configured in the hikyuu_XXX.ini file
my_tm.tocsv(sm.tmpdir())

View the csv with Excel, e.g.:

<img src="images/008_02_tocsv_look.png" align="left">

# 3 Use serialization to save or reload an existing TM object

In [ ]:
# Save to the specified file
from datetime import date
filename = "my_trade_record_{}.pkl".format(date.today())
hku_save(my_tm, filename)

In [ ]:
# Load the saved TM object
new_my_tm = hku_load(filename)

# 4 Use the order broker

In [ ]:
# Create a simulated trading account for backtesting with an initial capital of 300,000
my_tm = crtTM(init_cash=300000, date=Datetime(201701010000))

# Register the live trading order broker
ob = crtOB(TestOrderBroker())
my_tm.reg_broker(ob) # TestOerderBroker is a test order broker object that only prints
# Note: pybind does not support the following calling style; you must create the instance first and then pass it!!!
# my_tm.reg_broker(crtOB(TestOrderBroker(), False))

# Modify the last datetime of the order broker as needed; only after this datetime will the order broker actually issue the order instructions
my_tm.broker_last_datetime=Datetime(201701010000)

# Create the signal generator (the 5-day EMA as the fast line and the 10-day EMA of the 5-day EMA itself as the slow line; buy when the fast line crosses the slow line upward, and sell otherwise)
my_sg = SG_Flex(EMA(C, n=5), slow_n=10)

# Fixedly buy 1000 shares each time
my_mm = MM_FixedCount(1000)

# Create the trading system and run it
sys = SYS_Simple(tm = my_tm, sg = my_sg, mm = my_mm)
sys.run(sm['sz000001'], Query(-150))

In [ ]:
my_tm.get_trade_list().to_df()

In [ ]:
my_tm.get_trade_list().to_np()

In [ ]:
my_tm.get_history_position_list().to_df()

In [ ]:
my_tm.get_history_position_list().to_np()